# Quantile forecasting

### Installation

In [1]:
!pip install scikit-learn pandas numpy joblib


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importation of ```setup()``` method

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../src").resolve()))

from setup import setup

### Application of the method

I start by loading the dataset using my ```setup()``` function.

In [3]:
data = setup(
    csv_path="../../data/processed/financial_tools_datset.csv",
    target_col="Price",
    horizon=1
)

### Training and valuation splits

Here, I’m basically transforming raw market data into a supervised learning problem. I define my target as the future return, not the price itself. That’s important, because I care about movement, not absolute value.

I also split the data in time order, not randomly. If I shuffled this, I would leak future information into the past and completely destroy the validity of the model.

Finally, I normalize the features using StandardScaler. I do this so the model doesn’t get biased by scale differences across features.

👉 At this point, I have a clean mapping:

Inputs → market state

Output → future return

In [4]:
X_train = data["X_train"]
X_val   = data["X_val"]
X_test  = data["X_test"]

y_train = data["y_train"]
y_val   = data["y_val"]
y_test  = data["y_test"]

## The model + gradient boost

Now I train models to predict quantiles, not averages.

Instead of asking:

> “What will the return be?”

I’m asking:

> “What is a realistic lower bound, median, and upper bound for the return?”

Each model is trained with a different alpha:

0.1 → pessimistic scenario\
0.5 → median\
0.9 → optimistic scenario

So instead of a single prediction, I’m building a range of possible outcomes.

👉 This is the first step toward modeling uncertainty explicitly.

In [5]:
from sklearn.ensemble import GradientBoostingRegressor

quantiles = [0.1, 0.5, 0.9]
models = {}

for q in quantiles:
    model = GradientBoostingRegressor(
        loss="quantile",
        alpha=q,
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05
    )
    
    model.fit(X_train, y_train)
    models[q] = model

Here I simply run each trained model on the test set. Each model outputs a different view of the future:

1. One says “things could go this bad”

2. Another says “this is the most typical outcome”

Another says “this is the best-case scenario”

I store all of them separately because they each represent a different slice of the distribution.\
👉 I’m no longer dealing with a point estimate — I now have a distribution approximation.

In [6]:
preds = {}

for q, model in models.items():
    preds[q] = model.predict(X_test)

q10 = preds[0.1]
q50 = preds[0.5]
q90 = preds[0.9]

### Avaliation (Pinball Loss)

I evaluate each quantile using Pinball Loss. This is important: I cannot use MSE or anything like that. Those assume I’m predicting a mean. I’m not.

Each quantile has its own loss, because each one represents a different objective:

- Lower quantile → penalizes underestimation differently
- Upper quantile → penalizes overestimation differently

>👉 If I evaluate this the wrong way, I’ll think the model is good when it’s actually garbage.

In [7]:
import numpy as np

def pinball_loss(y_true, y_pred, q):
    return np.mean(np.maximum(q*(y_true - y_pred), (q-1)*(y_true - y_pred)))

for q in quantiles:
    loss = pinball_loss(y_test, preds[q], q)
    print(f"Quantile {q}: {loss:.6f}")

Quantile 0.1: 0.000693
Quantile 0.5: 0.001240
Quantile 0.9: 0.000862


What this tells me

- The values are low and consistent → no obvious instability
- q0.5 (median) has the highest loss → expected
    - It’s the hardest quantile to fit in noisy financial data
- q0.1 and q0.9 are lower → tails are being fit “easier”

### Interpretation

Our model is not broken. There’s no immediate red flag. Scale looks reasonable for return data.
> **Variable intervals → model captured heteroscedasticity**

### Calibration

**Generally expected: 0.8**

In [8]:
coverage = ((y_test >= q10) & (y_test <= q90)).mean()
print(f"Coverage [0.1–0.9]: {coverage:.3f}")

Coverage [0.1–0.9]: 0.782


Good. This is actually solid.

**Quick read**: 
- Expected: 0.8
- Got: 0.782

>👉 Error ≈ -0.018 (very small)

### Interpretation

- Slight undercoverage → model is a bit overconfident

    - **But**: Deviation is small!

Totally acceptable for a first baseline

### Risk envelope

Now I turn predictions into something operational.

I build an interval:

- Lower bound → q10
- Upper bound → q90

Then I compute the width of this interval. This gives me:
1. How uncertain the market is right now
2. How wide my expected outcomes are

**👉 This is directly usable for:**

1. position sizing
2. stop-loss distance
3. capital allocation

This is where the model stops being “ML” and becomes trading infrastructure.

In [9]:
import pandas as pd

df_results = pd.DataFrame({
    "y_true": y_test,
    "q10": q10,
    "q50": q50,
    "q90": q90
})

df_results["interval_width"] = df_results["q90"] - df_results["q10"]
df_results.head()

,y_true,q10,q50,q90,interval_width
0,0.000461,-0.002739,0.001173,0.006000,0.008740
1,0.001199,-0.002755,-0.000242,0.004849,0.007604
2,-0.003500,-0.004951,-0.002266,0.004544,0.009494
3,-0.000370,-0.001075,-0.000398,0.004702,0.005777
4,0.000925,-0.000664,0.000394,0.004783,0.005448


# Envelope signs derivation

Definition:

- long_signal = (q10 > 0)
- short_signal = (q90 < 0)

Meaning:\
- Long signal → even the pessimistic scenario (10th percentile) is positive
- Short signal → even the optimistic scenario (90th percentile) is negative

Interpretation:\
These are high-confidence directional trades. We are filtering only situations where the entire predicted distribution is on one side of zero. We built a strict decision filter based on distribution support, not probability.

In [10]:
df_results["long_signal"] = df_results["q10"] > 0
df_results["short_signal"] = df_results["q90"] < 0
print("Long signals:", df_results["long_signal"].sum())
print("Short signals:", df_results["short_signal"].sum())

Long signals: 19
Short signals: 1


### Expectation measure

Definition:

- ```expected_value = q50```
Meaning:\
The median of the predicted conditional distribution. Robust estimate of central tendency (less sensitive than mean).\
Interpretation:
- Represents the most likely return outcome
- Serves as your baseline directional bias

In [11]:
df_results["expected_value"] = df_results["q50"]
print(df_results["expected_value"].describe())

count    206.000000
mean       0.000139
std        0.002911
min       -0.008271
25%       -0.001809
50%        0.000120
75%        0.002028
max        0.010353
Name: expected_value, dtype: float64


### Risk-adjusted signal

Definition:
- risk_reward = q50 / (q90 - q10)
Meaning:\
Signal normalized by its uncertainty (interval width).

Interpretation:
- High value → strong signal + low uncertainty
- ow value → weak signal or high uncertainty

In [12]:
df_results["risk_reward"] = df_results["q50"] / df_results["interval_width"]
print(df_results["risk_reward"].describe())

count    206.000000
mean       0.023703
std        0.335906
min       -0.902080
25%       -0.195383
50%        0.013220
75%        0.213083
max        0.925684
Name: risk_reward, dtype: float64


### Acerto direcional

In [13]:
direction = (df_results["y_true"] > 0).astype(int)
pred_direction = (df_results["q50"] > 0).astype(int)

accuracy = (direction == pred_direction).mean()
print(f"Directional accuracy: {accuracy:.3f}")

Directional accuracy: 0.816


# PnL

In [14]:
df_results["position"] = 0
df_results.loc[df_results["long_signal"], "position"] = 1
df_results.loc[df_results["short_signal"], "position"] = -1

df_results["strategy_return"] = df_results["position"] * df_results["y_true"]

print("Total return:", df_results["strategy_return"].sum())
print("Mean return:", df_results["strategy_return"].mean())
print("Sharpe (proxy):", df_results["strategy_return"].mean() / df_results["strategy_return"].std())

Total return: 0.11455127316791225
Mean return: 0.000556074141591807
Sharpe (proxy): 0.2269912141969239


# LightGBM + more quantiles

In [15]:
!pip install lightgbm


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Training the model LightGBM

In [16]:
import lightgbm as lgb

quantiles = [0.05, 0.25, 0.5, 0.75, 0.95]
models = {}

for q in quantiles:
    model = lgb.LGBMRegressor(
        objective="quantile",
        alpha=q,
        n_estimators=300,
        learning_rate=0.03,
        max_depth=5
    )
    
    model.fit(X_train, y_train)
    models[q] = model

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002467 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5778
[LightGBM] [Info] Number of data points in the train set: 956, number of used features: 24
[LightGBM] [Info] Start training from score -0.007882
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

### Prediction

In [17]:
import pandas as pd

X_test_df = pd.DataFrame(X_test, columns=data["feature_columns"])

preds = {q: model.predict(X_test_df) for q, model in models.items()}

q05, q25, q50, q75, q95 = preds.values()

# Generating outputs for QF

## Final dataset of QF

In [18]:
df_qf = pd.DataFrame({
    "y_true": y_test,
    "q05": q05,
    "q25": q25,
    "q50": q50,
    "q75": q75,
    "q95": q95
})

### Width

In [19]:
df_qf["uncertainty"] = df_qf["q95"] - df_qf["q05"]

### Asymetry

In [20]:
df_qf["asymmetry"] = (df_qf["q95"] + df_qf["q05"] - 2*df_qf["q50"])

### Minimum output for RL

In [21]:
df_qf_final = df_qf[[
    "q05", "q50", "q95",
    "uncertainty",
    "asymmetry"
]]

### Visual outputs

In [23]:
print(df_qf_final.describe())

              q05         q50         q95  uncertainty   asymmetry
count  206.000000  206.000000  206.000000   206.000000  206.000000
mean    -0.004918    0.000107    0.005350     0.010268    0.000219
std      0.002197    0.003322    0.002452     0.002071    0.003610
min     -0.011736   -0.009000    0.000388     0.006122   -0.009315
25%     -0.006427   -0.002346    0.003428     0.008835   -0.002428
50%     -0.004932    0.000108    0.005140     0.009978    0.000076
75%     -0.003165    0.002467    0.006891     0.011370    0.002765
max     -0.001091    0.010509    0.013936     0.017873    0.008135


In [22]:
print("Mean uncertainty:", df_qf["uncertainty"].mean())
print("Mean asymmetry:", df_qf["asymmetry"].mean())

Mean uncertainty: 0.010268181747625036
Mean asymmetry: 0.00021875360803629053


### Minimum scope of paramaters

In [24]:
quantiles = [0.05, 0.25, 0.5, 0.75, 0.95]

model = lgb.LGBMRegressor(
    objective="quantile",
    n_estimators=300,
    learning_rate=0.03,
    max_depth=5
)